# 05: Modeling Features, Targets & Data Splits
**Milestone 2: Week 4 — Steps 9 to 12**

This notebook inspects the output generated by `src/build_features.py`:
- **Step 9:** Model task formulation and 75th percentile classification target
- **Step 10:** Feature table inspection (known in advance, fixed route, exact calendar lags)
- **Step 11:** Date splits inspection (Train: Jan-May, Val: Jun-Jul, Test: Aug-Sep)
- **Step 12:** Scikit-learn Pipeline with ColumnTransformer (`models/preprocessor.joblib`)

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.eda_utils import typical_days, DAY_TYPE_COLORS, calc_period_ratio, calc_cv

DATA = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
TAB = ROOT / "reports" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

matched = pd.read_parquet(DATA / "matched_hourly.parquet")
profiles = pd.read_parquet(DATA / "route_profiles.parquet")
print(f"Loaded matched hourly: {len(matched):,} rows (41 routes)")
print(f"Loaded route profiles: {len(profiles)} routes")


Loaded matched hourly: 233,227 rows (41 routes)
Loaded route profiles: 41 routes


## Step 9: Classification Target Formulation ('High Demand per Trip')
An hour counts as 'high demand per trip' when `riders_per_trip` exceeds the 75th percentile for that route and day type, calculated strictly from training dates (`date <= 2026-05-31`). It must be predicted without using current-hour ridership.

In [2]:
thresholds = pd.read_csv(TAB / 'm2_high_demand_thresholds.csv')
print(f'Total route x day_type threshold combinations: {len(thresholds)}')
print('\nSample 75th Percentile Thresholds (Training Period):')
thresholds.head(15)

Total route x day_type threshold combinations: 162

Sample 75th Percentile Thresholds (Training Period):


,route,day_type,threshold_p75
0,M1,Special/Excluded,9.512800
1,M1,Typical Saturday,18.760000
2,M1,Typical Sunday,18.000000
3,M1,Typical Weekday,17.194400
4,M10,Special/Excluded,7.070850
5,M10,Typical Saturday,11.280775
6,M10,Typical Sunday,9.736350
7,M10,Typical Weekday,12.195250
8,M101,Special/Excluded,12.638875
9,M101,Typical Saturday,18.710475


## Step 10: Features and Exact Calendar Lags
Lags (7d, 14d, 21d, 28d) are retrieved by exact calendar date lookup. Excluded hours remain `NaN` and are never 0-filled. A 1-day lag is omitted from the main model to prevent data leakage.

In [3]:
feat_all = pd.read_parquet(DATA / 'modeling_features_all.parquet')
lag_cols = ['lag_7d', 'lag_14d', 'lag_21d', 'lag_28d', 'lag_mean_available', 'lags_available_count']

print('Summary of Exact Calendar Lags:')
print(feat_all[lag_cols].describe().round(2))

print('\nSample row with available lags:')
feat_all[['route', 'date', 'hour', 'ridership', 'high_demand_per_trip'] + lag_cols].dropna().head(10)

Summary of Exact Calendar Lags:
          lag_7d    lag_14d    lag_21d    lag_28d  lag_mean_available  \
count  221044.00  219096.00  212939.00  206822.00           219591.00   
mean      250.47     250.31     249.16     249.09              249.01   
std       286.06     285.79     284.32     284.39              280.09   
min         0.00       0.00       0.00       0.00                0.00   
25%        41.00      41.00      41.00      41.00               42.75   
50%       150.00     150.00     149.00     149.00              151.50   
75%       377.00     377.00     376.00     376.00              376.00   
max      3456.00    3331.00    3331.00    3331.00             3135.50   

       lags_available_count  
count             233227.00  
mean                   3.69  
std                    0.89  
min                    0.00  
25%                    4.00  
50%                    4.00  
75%                    4.00  
max                    4.00  

Sample row with available lags:


,route,date,hour,ridership,high_demand_per_trip,lag_7d,lag_14d,lag_21d,lag_28d,lag_mean_available,lags_available_count
609,M1,2026-02-06,0,20,0,15.0,9.0,10.0,25.0,14.75,4
610,M1,2026-02-06,1,4,0,5.0,11.0,6.0,11.0,8.25,4
613,M1,2026-02-06,4,16,0,12.0,21.0,13.0,19.0,16.25,4
614,M1,2026-02-06,5,85,0,59.0,85.0,95.0,82.0,80.25,4
615,M1,2026-02-06,6,186,0,114.0,164.0,189.0,179.0,161.50,4
616,M1,2026-02-06,7,385,0,334.0,414.0,479.0,425.0,413.00,4
617,M1,2026-02-06,8,654,1,531.0,634.0,650.0,637.0,613.00,4
618,M1,2026-02-06,9,423,0,357.0,529.0,441.0,391.0,429.50,4
619,M1,2026-02-06,10,367,0,324.0,356.0,426.0,358.0,366.00,4
620,M1,2026-02-06,11,368,0,246.0,322.0,343.0,340.0,312.75,4


## Step 11: Chronological Date Splits
Data is partitioned chronologically with zero row shuffling:
- **Training:** 4 January to 31 May (125,445 rows)
- **Validation:** 1 June to 31 July (54,046 rows)
- **Test:** 1 August to 30 September (53,736 rows)

> **Known Seasonal Weakness:** Training data contains no summer months (July-August), while test data includes school reopenings in September.

In [4]:
splits_summary = pd.read_csv(TAB / 'm2_data_splits_summary.csv')
splits_summary

,set,start_date,end_date,rows,routes
0,Training,2026-01-04,2026-05-31,125445,41
1,Validation,2026-06-01,2026-07-31,54046,41
2,Test,2026-08-01,2026-09-30,53736,41
3,Total Window,2026-01-04,2026-09-30,233227,41


## Step 12: Scikit-Learn Preprocessing Pipeline
Inspects the fitted pipeline (`models/preprocessor.joblib`), which transforms routes, hour sine/cosine, and scales features.

In [5]:
import joblib

preprocessor = joblib.load(ROOT / 'models' / 'preprocessor.joblib')
train_df = pd.read_parquet(DATA / 'train_features.parquet').head(50)

X_transformed = preprocessor.transform(train_df)
print(f'Pipeline loaded successfully: {type(preprocessor).__name__}')
print(f'Sample input shape: {train_df.shape} -> Transformed feature dimension: {X_transformed.shape}')

Pipeline loaded successfully: ColumnTransformer
Sample input shape: (50, 34) -> Transformed feature dimension: (50, 77)
